# Soil-To-Plant Species Flow (Rare / Accessory / Core)

Species-level version of `Recall_genera_update_paper_results/soil_to_plant_genus_flow_rare_accessory_core.ipynb`.

Species come from `Oct_2026_recalled_species/lic16s_species_relabund.csv`. Taxa that couldn't be resolved to species keep a higher-rank label in the `Species` column (e.g. `g__Janthinobacterium_571130 Genus`, `f__SbA1 Family`) and are treated as one species each, as unresolved genera were in the genus table. Species under 0.05% of a sample (`AbundR == 0`) are dropped, which matches the cleaning of the recalled-genera table.

Species classes follow `core_and_rare_species_definitions_and_temporal_patterns.ipynb`. They are computed separately for plant and soil, using the seasonal samples only:

- **Core**: prevalence > 80% of samples
- **Rare**: prevalence < 10% of samples
- **Accessory**: everything in between

Prevalence = # unique samples where the species is present / total unique samples.

**Soil is done two ways, to compare:**
- **Pooled:** the cores from each timepoint are pooled into one sample (reads summed, then the 0.05% floor applied), as in the recalled-genera table (`t01_soil`, ...). Prevalence is across timepoints.
- **Individual cores:** each soil core is its own sample. Prevalence is across cores, so it's lower than pooled and more species come out rare.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from scipy.stats import fisher_exact

pd.options.display.max_columns = 50

In [2]:
### Recalled species table. Species under 0.05% of a sample (AbundR == 0) are
### dropped, as in the recalled-genera table
min_abundR = 1
### Plant samples missing from the recalled-genera table (LIC148, LIC162, LIC584 have
### < 1,600 16S reads; LIC577 unknown). Kept for now: see to_revisit.txt
samples_to_drop = []  # ["LIC148", "LIC162", "LIC577", "LIC584"]
species_table = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/Oct_2026_recalled_species/lic16s_species_relabund.csv"
)
species_table = species_table.loc[~species_table["Sample"].isin(samples_to_drop)].copy()
# No seascirc column in the species table: circadian timepoints start with "c"
species_table["seascirc"] = np.where(
    species_table["timepoint"].str.startswith("c"), "circadian", "seasonal"
)
species_table = species_table.rename(columns={"Sample": "sampID"})
seasonal = species_table["seascirc"] == "seasonal"

microbiome_abundance = species_table.loc[species_table["AbundR"] >= min_abundR]
long_term_microbiome = microbiome_abundance.loc[
    (microbiome_abundance["sample.type"] == "plant")
    & (microbiome_abundance["seascirc"] == "seasonal")
]
# Soil, individual cores
long_term_soil_microbiome = microbiome_abundance.loc[
    (microbiome_abundance["sample.type"] == "soil")
    & (microbiome_abundance["seascirc"] == "seasonal")
]

# Soil, pooled per timepoint: sum each species' reads over the timepoint's cores,
# divide by the summed depth, then apply the same floor (AbundR = rounded per-1000)
seasonal_soil = species_table.loc[
    (species_table["sample.type"] == "soil") & seasonal & (species_table["Abundance"] > 0)
]
pooled_depth = seasonal_soil.drop_duplicates("sampID").groupby("timepoint")["sampsums"].sum()
long_term_soil_microbiome_pooled = (
    seasonal_soil.assign(reads=seasonal_soil["Abundance"] / 1000 * seasonal_soil["sampsums"])
    .groupby(["timepoint", "Species"], as_index=False)["reads"]
    .sum()
)
long_term_soil_microbiome_pooled["Abundance"] = (
    long_term_soil_microbiome_pooled["reads"]
    / long_term_soil_microbiome_pooled["timepoint"].map(pooled_depth)
    * 1000
)
long_term_soil_microbiome_pooled = long_term_soil_microbiome_pooled.loc[
    long_term_soil_microbiome_pooled["Abundance"].round() >= min_abundR
].assign(sampID=lambda df: df["timepoint"] + "_soil")

print("plant samples:", long_term_microbiome["sampID"].nunique())
print("soil cores:", long_term_soil_microbiome["sampID"].nunique())
print("soil timepoints (pooled):", long_term_soil_microbiome_pooled["sampID"].nunique())

plant samples: 224
soil cores: 110
soil timepoints (pooled): 28


In [3]:
core_threshold = 0.80
rare_threshold = 0.10
unassigned_labels = {"unassigned", "na", "none", ""}


def classify_species(df, sample_col="sampID", tax_col="Species"):
    """Prevalence-based Rare / Accessory / Core classes (same rules as
    core_and_rare_species_definitions_and_temporal_patterns.ipynb)."""
    total_samples = df[sample_col].dropna().nunique()

    working = df[[sample_col, tax_col]].copy()
    working[tax_col] = working[tax_col].astype("string").str.strip()
    working = working[
        working[tax_col].notna()
        & ~working[tax_col].str.casefold().isin(unassigned_labels)
    ]

    out = (
        working.groupby(tax_col)[sample_col]
        .nunique()
        .rename("n_samples_present")
        .to_frame()
    )
    out["total_samples"] = total_samples
    out["prevalence"] = out["n_samples_present"] / total_samples
    out["classification"] = np.select(
        [out["prevalence"] > core_threshold, out["prevalence"] < rare_threshold],
        ["Core", "Rare"],
        default="Accessory",
    )
    return out.sort_values("prevalence", ascending=False)


plant_species_classification = classify_species(long_term_microbiome)
soil_approaches = {
    "pooled": "pooled_soil",
    "individual cores": "individual_soil_cores",
}
soil_species_classifications = {
    "pooled": classify_species(long_term_soil_microbiome_pooled),
    "individual cores": classify_species(long_term_soil_microbiome),
}

class_counts = pd.DataFrame(
    {
        "plant": plant_species_classification["classification"].value_counts(),
        **{
            f"soil ({approach})": classification["classification"].value_counts()
            for approach, classification in soil_species_classifications.items()
        },
    }
).reindex(["Rare", "Accessory", "Core"])
class_counts

,plant,soil (pooled),soil (individual cores)
classification,,,
Rare,388,157,409
Accessory,181,282,308
Core,40,128,67


### Pooled vs individual-core soil classes

In [4]:
# How each soil species is classed by the two soil approaches. A species can pass
# the 0.05% floor in a single core but fall below it once that timepoint's cores
# are pooled, so the individual cores have species the pooled table doesn't
soil_class_comparison = (
    soil_species_classifications["pooled"][["classification"]]
    .rename(columns={"classification": "pooled"})
    .join(
        soil_species_classifications["individual cores"][["classification"]].rename(
            columns={"classification": "individual cores"}
        ),
        how="outer",
    )
    .fillna("Not present")
)
pd.crosstab(
    soil_class_comparison["pooled"],
    soil_class_comparison["individual cores"],
    margins=True,
).reindex(
    index=["Rare", "Accessory", "Core", "Not present", "All"],
    columns=["Rare", "Accessory", "Core", "Not present", "All"],
    fill_value=0,
)

individual cores,Rare,Accessory,Core,Not present,All
pooled,,,,,
Rare,142,15,0,0,157
Accessory,50,232,0,0,282
Core,0,61,67,0,128
Not present,217,0,0,0,217
All,409,308,67,0,784


## Soil-To-Plant Flow

Tracks each soil species into one of four plant outcomes: Plant Rare, Plant Accessory, Plant Core, or Not Present in Plants. Species found in plants but not in soil are excluded. Done once for each soil approach.

Fisher's exact test (one-sided, `greater`) checks whether each soil class is enriched in each plant outcome, with all soil species as the background.

In [5]:
soil_classes = ["Rare", "Accessory", "Core"]
plant_classes = ["Rare", "Accessory", "Core", "Not Present in Plants"]

soil_to_plant_species_flows = {}
species_flow_counts = {}
for approach, soil_species_classification in soil_species_classifications.items():
    flow = (
        soil_species_classification[["classification"]]
        .rename(columns={"classification": "soil_classification"})
        .join(
            plant_species_classification[["classification"]].rename(
                columns={"classification": "plant_classification"}
            ),
            how="left",
        )
        .assign(
            plant_classification=lambda df: df["plant_classification"].fillna(
                "Not Present in Plants"
            )
        )
        .reset_index(names="Species")
    )
    soil_to_plant_species_flows[approach] = flow
    species_flow_counts[approach] = flow.groupby(
        ["soil_classification", "plant_classification"], as_index=False
    ).agg(species_count=("Species", "nunique"))

    print(f"Soil ({approach}):")
    display(
        species_flow_counts[approach]
        .pivot(
            index="soil_classification",
            columns="plant_classification",
            values="species_count",
        )
        .reindex(index=soil_classes, columns=plant_classes)
        .fillna(0)
        .astype(int)
    )

Soil (pooled):


plant_classification,Rare,Accessory,Core,Not Present in Plants
soil_classification,,,,
Rare,59,19,0,79
Accessory,125,81,7,69
Core,26,65,33,4


Soil (individual cores):


plant_classification,Rare,Accessory,Core,Not Present in Plants
soil_classification,,,,
Rare,122,34,0,253
Accessory,121,112,13,62
Core,8,32,27,0


In [6]:
species_enrichment_records = []

for approach, soil_to_plant_species_flow in soil_to_plant_species_flows.items():
    flow_counts = species_flow_counts[approach]
    species_background_total = len(soil_to_plant_species_flow)
    soil_species_class_totals = soil_to_plant_species_flow["soil_classification"].value_counts()
    plant_species_class_totals = soil_to_plant_species_flow[
        "plant_classification"
    ].value_counts()

    for soil_class in soil_classes:
        for plant_class in plant_classes:
            overlap = int(
                flow_counts.loc[
                    flow_counts["soil_classification"].eq(soil_class)
                    & flow_counts["plant_classification"].eq(plant_class),
                    "species_count",
                ].sum()
            )
            soil_total = int(soil_species_class_totals.get(soil_class, 0))
            plant_total = int(plant_species_class_totals.get(plant_class, 0))
            soil_not_plant = soil_total - overlap
            plant_not_soil = plant_total - overlap
            neither = species_background_total - overlap - soil_not_plant - plant_not_soil

            odds_ratio, p_value = fisher_exact(
                [[overlap, soil_not_plant], [plant_not_soil, neither]],
                alternative="greater",
            )

            species_enrichment_records.append(
                {
                    "soil_samples": approach,
                    "soil_classification": soil_class,
                    "plant_classification": plant_class,
                    "overlap_species": overlap,
                    "soil_class_total": soil_total,
                    "plant_class_total": plant_total,
                    "background_total_soil_species": species_background_total,
                    "odds_ratio": round(odds_ratio, 3),
                    "p_value_greater": p_value,
                }
            )

species_enrichment_results = pd.DataFrame(species_enrichment_records)
species_enrichment_results

,soil_samples,soil_classification,plant_classification,overlap_species,soil_class_total,plant_class_total,background_total_soil_species,odds_ratio,p_value_greater
0,pooled,Rare,Rare,59,157,210,567,1.033,4.712598e-01
1,pooled,Rare,Accessory,19,157,165,567,0.249,1.000000e+00
2,pooled,Rare,Core,0,157,40,567,0.000,1.000000e+00
3,pooled,Rare,Not Present in Plants,79,157,152,567,4.676,3.589552e-14
4,pooled,Accessory,Rare,125,282,210,567,1.873,2.367286e-04
5,pooled,Accessory,Accessory,81,282,165,567,0.964,6.137008e-01
6,pooled,Accessory,Core,7,282,40,567,0.194,9.999976e-01
7,pooled,Accessory,Not Present in Plants,69,282,152,567,0.788,9.108952e-01
8,pooled,Core,Rare,26,128,210,567,0.353,9.999988e-01
9,pooled,Core,Accessory,65,128,165,567,3.498,2.567088e-09


In [7]:
class_colors = {
    "Rare": "#d97d54",
    "Accessory": "#7a9e7e",
    "Core": "#30475e",
    "Not Present in Plants": "#b0b7c3",
}


def hex_to_rgba(hex_color, alpha):
    r, g, b = (int(hex_color.lstrip("#")[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"


def make_sankey(
    flow_df,
    node_labels,
    title_text,
    width=950,
    height=550,
    show_node_counts=False,
):
    """flow_df needs source, target, soil_classification, plant_classification
    and species_count columns. Nodes are coloured by class; links by soil class."""
    node_index = {label: index for index, label in enumerate(node_labels)}
    display_labels = node_labels

    source_counts = flow_df.groupby("source")["species_count"].sum()
    target_counts = flow_df.groupby("target")["species_count"].sum()
    node_totals = pd.concat([source_counts, target_counts], axis=1).max(axis=1)
    node_totals = node_totals.reindex(node_labels, fill_value=0).astype(int)
    if show_node_counts:
        display_labels = [f"{label} (n={node_totals[label]})" for label in node_labels]

    # Pin nodes top-to-bottom in node_labels order (soil left, plant right);
    # plotly's automatic layout otherwise shuffles them
    node_x, node_y = [], []
    for side in ("Soil ", None):
        column = [
            label for label in node_labels if label.startswith("Soil ") == bool(side)
        ]
        heights = node_totals[column] / node_totals[column].sum()
        gap = 0.04
        scale = 1 - gap * (len(column) - 1)
        tops = np.concatenate([[0], np.cumsum(heights * scale + gap)[:-1]])
        for label, top, node_height in zip(column, tops, heights * scale):
            node_x.append(0.001 if side else 0.999)
            node_y.append(top + node_height / 2)
    node_order = [label for label in node_labels if label.startswith("Soil ")] + [
        label for label in node_labels if not label.startswith("Soil ")
    ]
    position = dict(zip(node_order, zip(node_x, node_y)))

    node_colors = [
        class_colors[label.removeprefix("Soil ").removeprefix("Plant ")]
        for label in node_labels
    ]
    link_colors = [
        hex_to_rgba(class_colors[c], 0.35) for c in flow_df["soil_classification"]
    ]

    figure = go.Figure(
        data=[
            go.Sankey(
                arrangement="fixed",
                node=dict(
                    label=display_labels,
                    x=[position[label][0] for label in node_labels],
                    y=[position[label][1] for label in node_labels],
                    pad=18,
                    thickness=22,
                    color=node_colors,
                ),
                link=dict(
                    source=flow_df["source"].map(node_index).tolist(),
                    target=flow_df["target"].map(node_index).tolist(),
                    value=flow_df["species_count"].tolist(),
                    color=link_colors,
                    label=(
                        flow_df["soil_classification"]
                        + " -> "
                        + flow_df["plant_classification"]
                        + " | species: "
                        + flow_df["species_count"].astype(str)
                    ).tolist(),
                ),
            )
        ]
    )
    figure.update_layout(
        title_text=title_text,
        font_size=12,
        width=width,
        height=height,
    )
    return figure


def add_node_names(flow_df):
    flow_df = flow_df.copy()
    flow_df["source"] = "Soil " + flow_df["soil_classification"]
    flow_df["target"] = flow_df["plant_classification"].where(
        flow_df["plant_classification"].eq("Not Present in Plants"),
        "Plant " + flow_df["plant_classification"],
    )
    return flow_df

In [8]:
all_species_node_labels = (
    [f"Soil {c}" for c in soil_classes]
    + [f"Plant {c}" for c in soil_classes]
    + ["Not Present in Plants"]
)
figure_dir = Path("/Users/michael/Git/Outdoor_microbiome/Recalled_species_update_results/Figures")
figure_dir.mkdir(parents=True, exist_ok=True)

for approach, file_suffix in soil_approaches.items():
    all_species_fig = make_sankey(
        add_node_names(species_flow_counts[approach]),
        all_species_node_labels,
        f"Flow of Soil Species ({approach}) Into Plant Rare, Accessory, Core, or Absent Classes",
        width=1100,
        show_node_counts=True,
    )
    species_output_path = figure_dir / f"soil_to_plant_species_flow_rare_accessory_core_{file_suffix}.pdf"
    all_species_fig.write_image(species_output_path)
    print(f"Saved species-level Sankey figure to {species_output_path}")
    all_species_fig.show()

Saved species-level Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recalled_species_update_results/Figures/soil_to_plant_species_flow_rare_accessory_core_pooled_soil.pdf


Saved species-level Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recalled_species_update_results/Figures/soil_to_plant_species_flow_rare_accessory_core_individual_soil_cores.pdf


### Shared species only

Restricted to species found in both soil and plants (drops `Not Present in Plants`).

In [9]:
shared_species_node_labels = all_species_node_labels[:-1]

for approach in soil_approaches:
    flow_counts = species_flow_counts[approach]
    shared_species_flow = add_node_names(
        flow_counts.loc[flow_counts["plant_classification"].ne("Not Present in Plants")]
    )
    print(f"Soil ({approach}):")
    display(
        shared_species_flow.pivot(
            index="soil_classification",
            columns="plant_classification",
            values="species_count",
        )
        .reindex(index=soil_classes, columns=soil_classes)
        .fillna(0)
        .astype(int)
    )
    shared_species_fig = make_sankey(
        shared_species_flow,
        shared_species_node_labels,
        f"Flow of Shared Soil Species ({approach}) Into Plant Rare, Accessory, or Core Classes",
        show_node_counts=True,
    )
    shared_species_fig.show()

Soil (pooled):


plant_classification,Rare,Accessory,Core
soil_classification,,,
Rare,59,19,0
Accessory,125,81,7
Core,26,65,33


Soil (individual cores):


plant_classification,Rare,Accessory,Core
soil_classification,,,
Rare,122,34,0
Accessory,121,112,13
Core,8,32,27


### Rare and core only

Restricted to species that are rare or core in soil **and** rare or core in plants (drops Accessory on both sides and `Not Present in Plants`).

In [10]:
rare_core_classes = ["Rare", "Core"]
rare_core_node_labels = [f"Soil {c}" for c in rare_core_classes] + [
    f"Plant {c}" for c in rare_core_classes
]

for approach, file_suffix in soil_approaches.items():
    flow_counts = species_flow_counts[approach]
    rare_core_species_flow = add_node_names(
        flow_counts.loc[
            flow_counts["soil_classification"].isin(rare_core_classes)
            & flow_counts["plant_classification"].isin(rare_core_classes)
        ]
    )
    print(f"Soil ({approach}):")
    display(
        rare_core_species_flow.pivot(
            index="soil_classification",
            columns="plant_classification",
            values="species_count",
        )
        .reindex(index=rare_core_classes, columns=rare_core_classes)
        .fillna(0)
        .astype(int)
    )
    rare_core_species_fig = make_sankey(
        rare_core_species_flow,
        rare_core_node_labels,
        f"Flow of Soil ({approach}) Rare and Core Species Into Plant Rare or Core Classes",
        show_node_counts=True,
    )
    rare_core_output_path = figure_dir / f"soil_to_plant_species_flow_rare_core_only_{file_suffix}.pdf"
    rare_core_species_fig.write_image(rare_core_output_path)
    print(f"Saved rare/core-only Sankey figure to {rare_core_output_path}")
    rare_core_species_fig.show()

Soil (pooled):


plant_classification,Rare,Core
soil_classification,,
Rare,59,0
Core,26,33


Saved rare/core-only Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recalled_species_update_results/Figures/soil_to_plant_species_flow_rare_core_only_pooled_soil.pdf


Soil (individual cores):


plant_classification,Rare,Core
soil_classification,,
Rare,122,0
Core,8,27


Saved rare/core-only Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recalled_species_update_results/Figures/soil_to_plant_species_flow_rare_core_only_individual_soil_cores.pdf
